In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/mnt/home/test/beluga-call-pipeline/


In [2]:
import pandas as pd
import glob
import os

# Path to the snippets directory
snippets_dir = "../../data/evaluation_snippets/v2/"

# Get all folders in the snippets directory
snippet_folders = [f for f in os.listdir(snippets_dir) 
                   if os.path.isdir(os.path.join(snippets_dir, f))]

# Read all files from each folder's manual_verification directory
dfs = []
for folder in snippet_folders:
    manual_verification_dir = os.path.join(snippets_dir, folder, "manual_verification")
    
    # Check if manual_verification directory exists
    if not os.path.exists(manual_verification_dir):
        print(f"No manual_verification folder found in {folder}")
        continue
    
    # Get all files in the manual_verification directory
    manual_files = glob.glob(os.path.join(manual_verification_dir, "*"))
    
    # Read each file and add folder name as a column
    for file in manual_files:
        try:
            df = pd.read_csv(file, sep='\t', engine='python')
            df['site'] = folder  # Add folder name as a column
            context = df['context'].iloc[0]
            df["labeled_snippet_dir"] = os.path.join(snippets_dir, folder, context)

            # Extract annotator from filename: text between '.selections' and '.txt', remove leading character
            base = os.path.basename(file)
            if '.selections' in base and base.endswith('.txt'):
                annotator_raw = base.split('.selections')[1].replace('.txt', '')
                annotator = annotator_raw[1:] if len(annotator_raw) > 1 else ''
                df['annotator'] = annotator
                
            dfs.append(df)
            print(f"Read {os.path.basename(file)} from {folder}")
        except Exception as e:
            print(f"Could not read {file}: {e}")

# Concatenate all DataFrames into one
if dfs:
    all_manual_df = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(all_manual_df)}")
    print(f"Datasets: {all_manual_df['site'].unique()}")
else:
    print("No data files found")
    all_manual_df = pd.DataFrame()

Read 5725.200731140339.snippet.selections_JAA.txt from KAM_2020
Read 5725.200730123041.snippet.selections-ma.txt from KAM_2020
Read 5725.200730122420.snippet.selections-ma.txt from KAM_2020
Read 5725.200801152701.snippet.selections_EM.txt from KAM_2020
Read 5725.200730132804.snippet.selections-ma.txt from KAM_2020
Read 5725.200729102125.snippet.selections_JAA.txt from KAM_2020
Read 5725.200801153022.snippet.selections_EM.txt from KAM_2020
Read 5725.200801154645.snippet.selections_EM.txt from KAM_2020
Read 5725.200731141523.snippet.selections-ma.txt from KAM_2020
Read 5725.200801150025.snippet.selections-ma.txt from KAM_2020
Read 5725.200801152155.snippet.selections-ma.txt from KAM_2020
Read 5725.200801150830.snippet.selections_JAA.txt from KAM_2020
Read 5725.200730122111.snippet.selections_JAA.txt from KAM_2020
Read 5725.200731142245.snippet.selections_JAA.txt from KAM_2020
Read 5725.200729112727.snippet.selections_JAA.txt from KAM_2020
Read 5725.200730132426.snippet.selections-ma.txt 

In [3]:
labels_df = all_manual_df.copy()

labels_df = labels_df.drop(columns=["Selection", "View", "Channel", "Low Freq (Hz)", "High Freq (Hz)", "ECHO", "HFPC", "BBPC", "Whistle"])


In [4]:
labels_df["annotator"].value_counts(dropna=False)

annotator
EM     1680
JAA    1678
ma     1227
VA      836
Name: count, dtype: int64

In [5]:
labels_df.groupby("annotator")["GROUNDTRUTH"].value_counts(dropna=False)

annotator  GROUNDTRUTH
EM         NaN            1680
JAA        e               836
           a               466
           w               201
           ew              101
           eb               39
           eh               27
           b                 6
           h                 2
VA         NaN             783
           w                52
           b                 1
ma         e               546
           a               284
           w               243
           ew               51
           ew?              29
           eb               21
           eh               16
           ewb              12
           h                 9
           wb                5
           wh                2
           w?                2
           e                 1
           ewh               1
           ew?h              1
           ew?b              1
           wbh               1
           b                 1
           w?b               1
Name: count, dty

In [6]:
#TODO: Make sure this is still valide that they ddidnt' put a for absence, valeria in this case
labels_df["GROUNDTRUTH"].fillna("a", inplace=True)

/tmp/ipykernel_2242177/1128083399.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  labels_df["GROUNDTRUTH"].fillna("a", inplace=True)


0        e
1        e
2        e
3       ew
4       ew
        ..
5416     a
5417     a
5418     a
5419     a
5420     a
Name: GROUNDTRUTH, Length: 5421, dtype: object

In [7]:
labels_df = labels_df.rename(columns={"snippet_filename": "labeled_snippet_filename"})

In [8]:
from pipeline.pipeline import get_hydrophone_model
from data_preprocessing.spectrogram.spectrogram_generator import HYDROPHONE_SENSITIVITY

# Apply get_hydrophone_model to the original_filename column for all rows
labels_df["HydrophoneModel"] = labels_df["original_filename"].apply(get_hydrophone_model)
labels_df["HydrophoneSensitivity"] = labels_df["HydrophoneModel"].apply(HYDROPHONE_SENSITIVITY.get_sensitivity)

In [9]:
labels_df["HydrophoneSensitivity"].value_counts()

HydrophoneSensitivity
-172.7    2992
-175.7    2429
Name: count, dtype: int64

In [10]:
labels_df.tail()

,Begin Time (s),End Time (s),GROUNDTRUTH,DETAILS,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,boat_labeling_file,boat_labeling_file_id,context,site,labeled_snippet_dir,annotator,HydrophoneModel,HydrophoneSensitivity
5416,115.0,116.0,NaN,NaN,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035,1155,201359382.170724093002.Table.1.selections.txt,1,Boat,BSM_2017,../../data/evaluation_snippets/v2/BSM_2017/Boat,VA,201359382,-172.7
5417,116.0,117.0,NaN,NaN,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035,1155,201359382.170724093002.Table.1.selections.txt,1,Boat,BSM_2017,../../data/evaluation_snippets/v2/BSM_2017/Boat,VA,201359382,-172.7
5418,117.0,118.0,NaN,NaN,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035,1155,201359382.170724093002.Table.1.selections.txt,1,Boat,BSM_2017,../../data/evaluation_snippets/v2/BSM_2017/Boat,VA,201359382,-172.7
5419,118.0,119.0,NaN,NaN,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035,1155,201359382.170724093002.Table.1.selections.txt,1,Boat,BSM_2017,../../data/evaluation_snippets/v2/BSM_2017/Boat,VA,201359382,-172.7
5420,119.0,120.0,NaN,NaN,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035,1155,201359382.170724093002.Table.1.selections.txt,1,Boat,BSM_2017,../../data/evaluation_snippets/v2/BSM_2017/Boat,VA,201359382,-172.7


In [11]:
import pandas as pd

# Convert snippet_start_time to datetime and add the offset in seconds
labels_df["clip_start_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["Begin Time (s)"], unit='s')
labels_df["clip_end_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["End Time (s)"], unit='s')

In [12]:
labels_df.rename(columns={"site": "Site"}, inplace=True)
labels_df["Site"] = labels_df["Site"].str.split("_").str[0]

In [13]:
labels_df["clip_filename"] = labels_df["Site"] + "_" + labels_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"
# Check for duplicates in clip_filename
duplicate_clips = labels_df[labels_df.duplicated("clip_filename", keep=False)]
if not duplicate_clips.empty:
    print("Duplicates found in 'clip_filename':")
    display(duplicate_clips)
else:
    print("No duplicates found in 'clip_filename'.")

Duplicates found in 'clip_filename':


,Begin Time (s),End Time (s),GROUNDTRUTH,DETAILS,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,...,boat_labeling_file_id,context,Site,labeled_snippet_dir,annotator,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,clip_filename
427,0.0,1.0,NaN,NaN,NaN,5725.200801145954.wav,5725.200801152701.snippet.wav,2020-08-01 15:27:01,1627,1747,...,3,Boat,KAM,../../data/evaluation_snippets/v2/KAM_2020/Boat,EM,5725,-175.7,2020-08-01 15:27:01,2020-08-01 15:27:02,KAM_20200801_15270100.wav
428,0.0,1.0,NaN,NaN,NaN,5725.200801145954.wav,5725.200801152701.snippet.wav,2020-08-01 15:27:01,1627,1747,...,3,Boat,KAM,../../data/evaluation_snippets/v2/KAM_2020/Boat,EM,5725,-175.7,2020-08-01 15:27:01,2020-08-01 15:27:02,KAM_20200801_15270100.wav
429,1.0,2.0,NaN,NaN,NaN,5725.200801145954.wav,5725.200801152701.snippet.wav,2020-08-01 15:27:01,1627,1747,...,3,Boat,KAM,../../data/evaluation_snippets/v2/KAM_2020/Boat,EM,5725,-175.7,2020-08-01 15:27:02,2020-08-01 15:27:03,KAM_20200801_15270200.wav
430,1.0,2.0,NaN,NaN,NaN,5725.200801145954.wav,5725.200801152701.snippet.wav,2020-08-01 15:27:01,1627,1747,...,3,Boat,KAM,../../data/evaluation_snippets/v2/KAM_2020/Boat,EM,5725,-175.7,2020-08-01 15:27:02,2020-08-01 15:27:03,KAM_20200801_15270200.wav
431,2.0,3.0,NaN,NaN,NaN,5725.200801145954.wav,5725.200801152701.snippet.wav,2020-08-01 15:27:01,1627,1747,...,3,Boat,KAM,../../data/evaluation_snippets/v2/KAM_2020/Boat,EM,5725,-175.7,2020-08-01 15:27:03,2020-08-01 15:27:04,KAM_20200801_15270300.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5176,117.0,118.0,NaN,NaN,NaN,201359382.170725063002.wav,201359382.170725070119.snippet.wav,2017-07-25 07:01:19,1877,1997,...,3,Boat,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,EM,201359382,-172.7,2017-07-25 07:03:16,2017-07-25 07:03:17,BSM_20170725_07031600.wav
5177,118.0,119.0,NaN,NaN,NaN,201359382.170725063002.wav,201359382.170725070119.snippet.wav,2017-07-25 07:01:19,1877,1997,...,3,Boat,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,EM,201359382,-172.7,2017-07-25 07:03:17,2017-07-25 07:03:18,BSM_20170725_07031700.wav
5178,118.0,119.0,NaN,NaN,NaN,201359382.170725063002.wav,201359382.170725070119.snippet.wav,2017-07-25 07:01:19,1877,1997,...,3,Boat,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,EM,201359382,-172.7,2017-07-25 07:03:17,2017-07-25 07:03:18,BSM_20170725_07031700.wav
5179,119.0,120.0,NaN,NaN,NaN,201359382.170725063002.wav,201359382.170725070119.snippet.wav,2017-07-25 07:01:19,1877,1997,...,3,Boat,BSM,../../data/evaluation_snippets/v2/BSM_2017/Boat,EM,201359382,-172.7,2017-07-25 07:03:18,2017-07-25 07:03:19,BSM_20170725_07031800.wav


In [14]:
def set_verif_flags(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


In [15]:
labels_df["BBPC"].value_counts()

BBPC
0    5333
1      88
Name: count, dtype: int64

## Clipping to 1 second audio files

In [16]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs"
os.makedirs(output_dir, exist_ok=True)

# Group by snippet to load each file only once
grouped = labels_df.groupby(["labeled_snippet_dir", "labeled_snippet_filename"])

for (snippet_dir, snippet_filename), group in tqdm(grouped, total=len(grouped)):
    # Build the full path to the source snippet
    source_path = os.path.join(snippet_dir, snippet_filename)
    
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["Begin Time (s)"] * sr)
            end_sample = int(row["End Time (s)"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")

  0%|          | 0/39 [00:00<?, ?it/s]/mnt/home/test/beluga-call-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 39/39 [00:49<00:00,  1.27s/it]


In [17]:
labels_df["Boat"] = labels_df["context"].apply(lambda x: 1 if x == "Boat" else 0)
labels_df = labels_df.drop(columns=["context"])


In [18]:
labels_df["Boat"].value_counts()

Boat
1    4229
0    1192
Name: count, dtype: int64

In [19]:
labels_output_dir = "../../data/Verified_Dataset/labels"

labels_df.to_csv(os.path.join(labels_output_dir, "labels_eval_v2.csv"), index=False)